# Black-Scholes Model

The Black-Scholes Model, or the Black-Scholes-Merton Model, is a mathematical model for the dynamics of a financial market containing derivative investment instruments.

## Black-Scholes Equation

The Black-Scholes Equation is a parabolic partial differential equation that expresses the price $V(S,t)$ of an option.

The equation is given by
$$
\frac{\partial V}{\partial t} + \frac{1}{2}\sigma^2S^2\frac{\partial^2 V}{\partial S^2} + rS\frac{\partial V}{\partial S} - rV = 0
$$
where
- $V$ is the value of an option
- $S$ is the price of the underlying asset
- $t$ is time
- $r$ is the risk-free interest rate
- $\sigma$ is the volatility of the underlying asset

## Black-Scholes Formula

The Black-Scholes formula is used to calculate the fair price or theoretical value of an options contract.

The formula for a European call option is given by
$$
C = SN(d_1) - Ke^{-rT}N(d_2)
$$
while the formula for a European put option is given by
$$
P = Ke^{-rT}N(-d_2) - SN(-d_1)
$$
where
$$
d_1 = \frac{\ln(\frac{S}{K})+(r+\frac{\sigma^2}{2})T}{\sigma\sqrt{T}}, \quad
d_2 = \frac{\ln(\frac{S}{K})+(r-\frac{\sigma^2}{2})T}{\sigma\sqrt{T}}
$$
and
- $C$ is the price of a European call option
- $P$ is the price of a European put option
- $S$ is the current price of the underlying asset
- $K$ is the strike price of the option
- $T$ is the time to expiration of the option in years
- $r$ is the risk-free interest rate
- $\sigma$ is the volatility of the underlying asset


In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import ipywidgets as widgets
from ipywidgets import interactive_output, FloatSlider, VBox, HBox
from IPython.display import display, HTML

from src.models.black_scholes import BlackScholes, implied_vol

## Interactive Pricer

In [2]:
# --- palette ---
C = {
    "ground":  "#EFF3F8",
    "surface": "#FFFFFF",
    "text":    "#1A2332",
    "muted":   "#64748B",
    "border":  "#CBD5E1",
    "grid":    "#E8EDF4",
    "call":    "#1D6FE8",
    "put":     "#D94F2B",
    "strike":  "#F59E0B",
}

# --- sliders ---
def make_slider(desc, val, lo, hi, step, fmt=None):
    kw = dict(value=val, min=lo, max=hi, step=step, description=desc,
              style={"description_width": "80px"},
              layout=widgets.Layout(width="320px"))
    if fmt:
        kw["readout_format"] = fmt
    return FloatSlider(**kw)

s_sl   = make_slider("Spot  S",   100,  50,  150, 1)
k_sl   = make_slider("Strike  K", 100,  50,  150, 1)
t_sl   = make_slider("Expiry  T", 1.0, 0.05, 2.0, 0.05, ".2f")
r_sl   = make_slider("Rate  r",  0.05, 0.00, 0.10, 0.005, ".3f")
sig_sl = make_slider("Vol  σ",   0.20, 0.05, 0.80, 0.01,  ".2f")

controls = dict(S=s_sl, K=k_sl, T=t_sl, r=r_sl, sigma=sig_sl)

# --- rc context ---
RC = {
    "figure.facecolor":  C["ground"],
    "axes.facecolor":    C["surface"],
    "axes.edgecolor":    C["border"],
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "grid.color":        C["grid"],
    "grid.linewidth":    0.7,
    "axes.grid":         True,
    "text.color":        C["text"],
    "axes.labelcolor":   C["muted"],
    "xtick.color":       C["muted"],
    "ytick.color":       C["muted"],
    "xtick.labelsize":   9,
    "ytick.labelsize":   9,
    "axes.labelsize":    10,
    "axes.titlesize":    12,
    "axes.titleweight":  "bold",
    "axes.titlecolor":   C["text"],
    "legend.framealpha": 0.95,
    "legend.edgecolor":  C["border"],
    "legend.fontsize":   8,
}

# --- greek card ---
def greek_card(sym, name, cv, pv, deriv):
    return f"""
    <div style="background:{C['surface']};border-radius:8px;
                padding:14px 16px;flex:1;min-width:118px;max-width:175px;
                border:1px solid {C['border']};border-top:3px solid {C['text']}">
      <div style="display:flex;align-items:baseline;gap:5px;margin-bottom:10px">
        <span style="font-size:20px;font-weight:700;color:{C['text']};
                     font-family:Georgia,'Times New Roman',serif">{sym}</span>
        <span style="font-size:10px;color:{C['muted']};letter-spacing:.7px;
                     text-transform:uppercase">{name}</span>
      </div>
      <div style="display:flex;gap:14px;margin-bottom:10px">
        <div>
          <div style="font-size:8.5px;font-weight:700;color:{C['call']};
                      letter-spacing:.9px;margin-bottom:3px">CALL</div>
          <div style="font-family:'Courier New',Courier,monospace;font-size:14px;
                      font-weight:700;color:{C['text']}">{cv}</div>
        </div>
        <div>
          <div style="font-size:8.5px;font-weight:700;color:{C['put']};
                      letter-spacing:.9px;margin-bottom:3px">PUT</div>
          <div style="font-family:'Courier New',Courier,monospace;font-size:14px;
                      font-weight:700;color:{C['text']}">{pv}</div>
        </div>
      </div>
      <div style="font-family:'Courier New',Courier,monospace;font-size:9px;
                  color:{C['muted']};border-top:1px solid {C['border']};
                  padding-top:7px">{deriv}</div>
    </div>"""

# --- update ---
def update(S, K, T, r, sigma):
    bs   = BlackScholes(S, K, T, r, sigma)
    spot = np.linspace(max(1, S * 0.5), S * 1.5, 400)
    cp   = np.array([BlackScholes(s, K, T, r, sigma).call_price() for s in spot])
    pp   = np.array([BlackScholes(s, K, T, r, sigma).put_price()  for s in spot])
    ci   = np.maximum(spot - K, 0)
    pi   = np.maximum(K - spot, 0)
    call_p, put_p = bs.call_price(), bs.put_price()

    with plt.rc_context(RC):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
        fig.patch.set_facecolor(C["ground"])
        fig.subplots_adjust(wspace=0.3)

        for ax, prices, intr, label, color, price, otype, leg_loc in [
            (ax1, cp, ci, "Call Option", C["call"], call_p, "call", "upper left"),
            (ax2, pp, pi, "Put Option",  C["put"],  put_p,  "put",  "upper right"),
        ]:
            ax.fill_between(spot, intr, prices, color=color, alpha=0.07, zorder=1)
            ax.plot(spot, intr,   color=color, lw=1.2, ls="--", alpha=0.4,
                    label="Intrinsic value", zorder=2)
            ax.plot(spot, prices, color=color, lw=2.5,
                    label="Model price", zorder=3)
            ax.axvline(K, color=C["strike"], lw=1.2, ls="-.", alpha=0.85,
                       label=f"Strike  {K:.0f}", zorder=4)
            ax.axvline(S, color=C["muted"],  lw=0.9, ls=":",  alpha=0.65,
                       label=f"Spot  {S:.0f}",   zorder=4)
            ax.scatter([S], [price], color=color, s=72, zorder=6,
                       edgecolors=C["surface"], linewidths=2)
            ax.annotate(f" ${price:.2f}", xy=(S, price),
                        xytext=(6, 0), textcoords="offset points",
                        fontsize=10, fontweight="bold",
                        color=color, va="center")
            ax.set_title(label)
            ax.set_xlabel("Spot price")
            ax.set_ylabel("Price ($)")
            ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
            ax.legend(loc=leg_loc)

        fig.suptitle(
            f"S = {S:.0f}   ·   K = {K:.0f}   ·   T = {T:.2f} yr   ·   r = {r:.1%}   ·   σ = {sigma:.0%}",
            fontsize=10.5, color=C["muted"], y=1.02,
        )
        plt.tight_layout()
        plt.show()

    # --- greek panel ---
    greeks = [
        ("Δ", "Delta",
         f"{bs.delta('call'):+.4f}", f"{bs.delta('put'):+.4f}", "∂V / ∂S"),
        ("Γ", "Gamma",
         f"{bs.gamma():.4f}", f"{bs.gamma():.4f}", "∂²V / ∂S²"),
        ("Θ", "Theta",
         f"{bs.theta('call'):+.4f}", f"{bs.theta('put'):+.4f}", "∂V / ∂t  (per day)"),
        ("ν", "Vega",
         f"{bs.vega():+.4f}", f"{bs.vega():+.4f}", "∂V / ∂σ  (per 1%)"),
        ("ρ", "Rho",
         f"{bs.rho('call'):+.4f}", f"{bs.rho('put'):+.4f}", "∂V / ∂r  (per 1%)"),
    ]

    price_bar = f"""
    <div style="display:flex;gap:10px;margin-bottom:12px">
      <div style="background:{C['call']};color:#fff;border-radius:6px;
                  padding:9px 20px;font-size:13px;font-weight:600;
                  font-family:-apple-system,'Segoe UI',sans-serif">
        Call &nbsp;
        <span style="font-family:'Courier New',monospace;font-size:15px">
          ${call_p:.4f}</span>
      </div>
      <div style="background:{C['put']};color:#fff;border-radius:6px;
                  padding:9px 20px;font-size:13px;font-weight:600;
                  font-family:-apple-system,'Segoe UI',sans-serif">
        Put &nbsp;
        <span style="font-family:'Courier New',monospace;font-size:15px">
          ${put_p:.4f}</span>
      </div>
    </div>"""

    cards = "".join(greek_card(*g) for g in greeks)

    display(HTML(f"""
    <div style="font-family:-apple-system,'Segoe UI',sans-serif;
                background:{C['ground']};padding:14px 2px 6px">
      {price_bar}
      <div style="display:flex;gap:8px;flex-wrap:wrap">{cards}</div>
    </div>"""))


# --- layout ---
header = widgets.HTML(f"""
<div style="font-family:-apple-system,'Segoe UI',sans-serif;
            padding:14px 4px 12px;border-bottom:1px solid {C['border']};
            margin-bottom:14px">
  <div style="font-size:10px;color:{C['muted']};letter-spacing:1.4px;
              text-transform:uppercase;margin-bottom:5px">
    Options Pricing  ·  European Exercise
  </div>
  <div style="font-size:22px;font-weight:700;color:{C['text']};
              letter-spacing:-.3px">
    Black–Scholes Interactive Pricer
  </div>
</div>""")

row1 = HBox([s_sl, k_sl, t_sl], layout=widgets.Layout(gap="12px", margin="0 0 6px"))
row2 = HBox([r_sl, sig_sl],     layout=widgets.Layout(gap="12px", margin="0 0 16px"))
out  = interactive_output(update, controls)

display(VBox([header, row1, row2, out],
             layout=widgets.Layout(max_width="960px")))

## Greek Curves

In [3]:
# --- sliders ---
s_sl_g   = make_slider("Spot  S",   100,  50,  150, 1)
k_sl_g   = make_slider("Strike  K", 100,  50,  150, 1)
t_sl_g   = make_slider("Expiry  T", 1.0, 0.05, 2.0, 0.05, ".2f")
r_sl_g   = make_slider("Rate  r",  0.05, 0.00, 0.10, 0.005, ".3f")
sig_sl_g = make_slider("Vol  σ",   0.20, 0.05, 0.80, 0.01,  ".2f")

controls_g = dict(S=s_sl_g, K=k_sl_g, T=t_sl_g, r=r_sl_g, sigma=sig_sl_g)

SHARED = "#7C3AED"  # call = put (Gamma, Vega)

# --- greek curves update ---
def update_greeks(S, K, T, r, sigma):
    spot = np.linspace(max(1, S * 0.5), S * 1.5, 400)
    bs_S = BlackScholes(S, K, T, r, sigma)

    def curve(fn):
        return np.array([fn(BlackScholes(s, K, T, r, sigma)) for s in spot])

    panels = [
        ("Δ", "Delta",
         curve(lambda b: b.delta("call")), curve(lambda b: b.delta("put")),
         bs_S.delta("call"), bs_S.delta("put"), True),
        ("Γ", "Gamma",
         curve(lambda b: b.gamma()), None,
         bs_S.gamma(), None, False),
        ("Θ", "Theta",
         curve(lambda b: b.theta("call")), curve(lambda b: b.theta("put")),
         bs_S.theta("call"), bs_S.theta("put"), True),
        ("ν", "Vega",
         curve(lambda b: b.vega()), None,
         bs_S.vega(), None, False),
        ("ρ", "Rho",
         curve(lambda b: b.rho("call")), curve(lambda b: b.rho("put")),
         bs_S.rho("call"), bs_S.rho("put"), True),
    ]

    with plt.rc_context(RC):
        fig, axes = plt.subplots(2, 3, figsize=(13, 7))
        fig.patch.set_facecolor(C["ground"])
        axes[1, 2].set_visible(False)

        for ax, (sym, name, c_arr, p_arr, c_s, p_s, split) in zip(axes.flat, panels):
            if split:
                ax.plot(spot, c_arr, color=C["call"], lw=2, label="Call")
                ax.plot(spot, p_arr, color=C["put"],  lw=2, label="Put")
                ax.scatter([S], [c_s], color=C["call"], s=55, zorder=5,
                           edgecolors=C["surface"], linewidths=1.5)
                ax.scatter([S], [p_s], color=C["put"],  s=55, zorder=5,
                           edgecolors=C["surface"], linewidths=1.5)
                ax.legend(fontsize=7.5)
            else:
                ax.plot(spot, c_arr, color=SHARED, lw=2, label="Call = Put")
                ax.scatter([S], [c_s], color=SHARED, s=55, zorder=5,
                           edgecolors=C["surface"], linewidths=1.5)
                ax.legend(fontsize=7.5)

            ax.axvline(K, color=C["strike"], lw=1,   ls="-.", alpha=0.75, zorder=3)
            ax.axvline(S, color=C["muted"],  lw=0.8, ls=":",  alpha=0.60, zorder=3)
            ax.axhline(0, color=C["border"], lw=0.8, alpha=0.8)
            ax.set_title(f"{sym}  {name}", pad=8)
            ax.set_xlabel("Spot")

        fig.suptitle(
            f"S = {S:.0f}   ·   K = {K:.0f}   ·   T = {T:.2f} yr   ·   r = {r:.1%}   ·   σ = {sigma:.0%}",
            fontsize=10.5, color=C["muted"], y=1.01,
        )
        plt.tight_layout()
        plt.show()


# --- layout ---
header_g = widgets.HTML(f"""
<div style="font-family:-apple-system,'Segoe UI',sans-serif;
            padding:14px 4px 12px;border-bottom:1px solid {C['border']};
            margin-bottom:14px">
  <div style="font-size:10px;color:{C['muted']};letter-spacing:1.4px;
              text-transform:uppercase;margin-bottom:5px">
    Sensitivity Analysis  ·  European Exercise
  </div>
  <div style="font-size:22px;font-weight:700;color:{C['text']};
              letter-spacing:-.3px">
    Greek Curves
  </div>
</div>""")

row1_g = HBox([s_sl_g, k_sl_g, t_sl_g], layout=widgets.Layout(gap="12px", margin="0 0 6px"))
row2_g = HBox([r_sl_g, sig_sl_g],        layout=widgets.Layout(gap="12px", margin="0 0 16px"))
out_g  = interactive_output(update_greeks, controls_g)

display(VBox([header_g, row1_g, row2_g, out_g],
             layout=widgets.Layout(max_width="960px")))

## Implied Volatility

In [4]:
# --- sliders ---
s_sl_iv  = make_slider("Spot  S",   100,  50,  150, 1)
k_sl_iv  = make_slider("Strike  K", 100,  50,  150, 1)
t_sl_iv  = make_slider("Expiry  T", 1.0, 0.05, 2.0, 0.05, ".2f")
r_sl_iv  = make_slider("Rate  r",  0.05, 0.00, 0.10, 0.005, ".3f")
mp_sl_iv = make_slider("Mkt Price", 10.0, 0.01, 50.0, 0.01, ".2f")

controls_iv = dict(S=s_sl_iv, K=k_sl_iv, T=t_sl_iv, r=r_sl_iv, market_price=mp_sl_iv)

# --- iv update ---
def update_iv(S, K, T, r, market_price):
    iv_call = implied_vol(market_price, S, K, T, r, "call")
    iv_put  = implied_vol(market_price, S, K, T, r, "put")

    with plt.rc_context(RC):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
        fig.patch.set_facecolor(C["ground"])

        for ax, otype, iv, color, leg_loc in [
            (ax1, "call", iv_call, C["call"], "upper left"),
            (ax2, "put",  iv_put,  C["put"],  "upper left"),
        ]:
            # centre x-axis on iv: guarantee iv is the exact midpoint
            if iv is not None:
                half   = max(0.1, iv * 0.5)
                lo     = max(0.001, iv - half)
                hi     = 2 * iv - lo   # mirror lo around iv → iv always at centre
                sigmas = np.linspace(lo, hi, 400)
            else:
                sigmas = np.linspace(0.01, 1.5, 400)

            prices    = np.array([
                BlackScholes(S, K, T, r, s).call_price() if otype == "call"
                else BlackScholes(S, K, T, r, s).put_price()
                for s in sigmas
            ])
            intrinsic = max(S - K, 0) if otype == "call" else max(K - S, 0)

            ax.fill_between(sigmas, intrinsic, prices, color=color, alpha=0.07, zorder=1)
            ax.plot(sigmas, prices, color=color, lw=2.5,
                    label=otype.capitalize(), zorder=2)
            ax.axhline(market_price, color=C["text"], lw=1.2, ls="--", alpha=0.7,
                       label=f"Market  ${market_price:.2f}", zorder=3)

            if iv is not None:
                iv_price = (BlackScholes(S, K, T, r, iv).call_price() if otype == "call"
                            else BlackScholes(S, K, T, r, iv).put_price())
                ax.axvline(iv, color=color, lw=1, ls=":", alpha=0.7, zorder=4)
                ax.scatter([iv], [iv_price], color=color, s=72, zorder=5,
                           edgecolors=C["surface"], linewidths=2)
                ax.annotate(f"  σ* = {iv:.1%}", xy=(iv, iv_price),
                            xytext=(6, -8), textcoords="offset points",
                            fontsize=9, fontweight="bold", color=color, va="top",
                            zorder=6)

            ax.set_title(f"{otype.capitalize()} Option")
            ax.set_xlabel("Volatility σ")
            ax.set_ylabel("Option price ($)")
            ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
            ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
            ax.legend(loc=leg_loc)

        fig.suptitle(
            f"S = {S:.0f}   ·   K = {K:.0f}   ·   T = {T:.2f} yr   ·   r = {r:.1%}   ·   Mkt = ${market_price:.2f}",
            fontsize=10.5, color=C["muted"],
        )
        plt.tight_layout(rect=[0, 0, 1, 0.93])
        plt.show()

    # --- iv panel ---
    def iv_badge(label, iv, color):
        val = f"{iv:.4f}" if iv is not None else "—"
        return f"""
        <div style="background:{color};color:#fff;border-radius:6px;
                    padding:9px 20px;font-size:13px;font-weight:600;
                    font-family:-apple-system,'Segoe UI',sans-serif">
          σ* {label} &nbsp;
          <span style="font-family:'Courier New',monospace;font-size:15px">{val}</span>
        </div>"""

    display(HTML(f"""
    <div style="font-family:-apple-system,'Segoe UI',sans-serif;
                background:{C['ground']};padding:14px 2px 6px">
      <div style="display:flex;gap:10px;margin-bottom:4px">
        {iv_badge("Call", iv_call, C["call"])}
        {iv_badge("Put",  iv_put,  C["put"])}
      </div>
    </div>"""))


# --- layout ---
header_iv = widgets.HTML(f"""
<div style="font-family:-apple-system,'Segoe UI',sans-serif;
            padding:14px 4px 12px;border-bottom:1px solid {C['border']};
            margin-bottom:14px">
  <div style="font-size:10px;color:{C['muted']};letter-spacing:1.4px;
              text-transform:uppercase;margin-bottom:5px">
    Inverse Problem  ·  European Exercise
  </div>
  <div style="font-size:22px;font-weight:700;color:{C['text']};
              letter-spacing:-.3px">
    Implied Volatility
  </div>
</div>""")

row1_iv = HBox([s_sl_iv, k_sl_iv, t_sl_iv], layout=widgets.Layout(gap="12px", margin="0 0 6px"))
row2_iv = HBox([r_sl_iv, mp_sl_iv],          layout=widgets.Layout(gap="12px", margin="0 0 16px"))
out_iv  = interactive_output(update_iv, controls_iv)

display(VBox([header_iv, row1_iv, row2_iv, out_iv],
             layout=widgets.Layout(max_width="960px")))